# 01 | Dựng bảng huấn luyện V6: bốn domain, có nhãn hard negative

**Mục đích.** Gộp đặc trưng 14 chiều của bốn nguồn thành một bảng duy nhất để notebook 02 huấn luyện V6C. Bản đọc được của `notebooks/v6/scripts/build_v6b_clean_manifest.py`.

| | |
|---|---|
| Đầu vào | `data/features/v6/deepbeat_train_v6_14f.csv`, `data/features/v6/mimic_perform_v6_14f.csv`, `data/splits/{pulsewatch,liu}_train.csv` |
| Đầu ra | `data/features/v7/healthsense_af_v6c_pacaware_train.csv` |
| Nhãn | `label_v7`: 1 = rung nhĩ, 0 = còn lại. `class_name` giữ nhóm gốc: AF, NSR, PAC_PVC, NON_AF, OTHER |

Chỉ phần **train** của mỗi domain được đưa vào; bệnh nhân DEV và TEST nằm ở file khác và không bao giờ chạm vào bảng này. Nhãn `class_name` là thứ cho phép V6 coi PAC/PVC là lớp âm tính khó khi huấn luyện.

In [1]:
# Tìm gốc repo và nạp thư viện src/healthsense_ml (chạy được từ bất kỳ thư mục con nào)
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_ml)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
FEATURES = ["HR_mean", "Mean_NN", "SDNN", "RMSSD", "NN50", "pNN50", "CV",
            "HF", "Total_Power", "HF_norm", "SD1", "SD2", "SampEn", "PPG_AC"]
COLS = FEATURES + ["label_v7", "domain", "subject_id", "class_name"]
SPLITS = ROOT / "data" / "splits"
OUT = ROOT / "data" / "features" / "v7" / "healthsense_af_v6c_pacaware_train.csv"
OVERWRITE = False   # True: ghi đè bảng đã dùng để train bản chuẩn; mặc định ghi ra *_rebuilt.csv để so sánh
frames = []

## 1. DeepBeat và MIMIC: nhãn nhị phân có sẵn

In [3]:
deep = pd.read_csv(ROOT / "data/features/v6/deepbeat_train_v6_14f.csv")
deep["domain"] = "DEEPBEAT"
deep["label_v7"] = deep["label"].astype(int)
deep["class_name"] = deep["label_v7"].map({0: "NON_AF", 1: "AF"})
frames.append(deep[COLS])

mimic = pd.read_csv(ROOT / "data/features/v6/mimic_perform_v6_14f.csv")
mimic["domain"] = "MIMIC_PERFORM"
mimic["label_v7"] = mimic["status"].astype(int)
mimic["subject_id"] = "mimic_" + mimic["record_id"].astype(str)
mimic["class_name"] = mimic["label_v7"].map({0: "NON_AF", 1: "AF"})
frames.append(mimic[COLS])
print("DeepBeat:", len(deep), "| MIMIC:", len(mimic))

DeepBeat: 19842 | MIMIC: 4130


## 2. PulseWatch và Liu: giữ nhóm nhịp gốc

PulseWatch có sẵn `class_name` với ba nhóm AF, NSR, PAC_PVC; Liu có cột `rhythm` nhiều loại, mọi thứ không phải AF gộp thành OTHER. Cách gộp này là điểm khác của V6 so với V5: ngoại tâm thu không bị trộn lẫn vào "bình thường".

In [4]:
pulse = pd.read_csv(SPLITS / "pulsewatch_train.csv")
pulse["domain"] = "PULSEWATCH"
pulse["label_v7"] = (pulse["class_name"] == "AF").astype(int)
pulse["subject_id"] = pulse["uid"].astype(str)
frames.append(pulse[COLS])

liu = pd.read_csv(SPLITS / "liu_train.csv")
liu["domain"] = "LIU"
liu["label_v7"] = (liu["rhythm"] == "AF").astype(int)
liu["subject_id"] = liu["subject_id"].astype(str)
liu["class_name"] = liu["label_v7"].map({0: "OTHER", 1: "AF"})
frames.append(liu[COLS])
print("PulseWatch:", len(pulse), "| Liu:", len(liu))

PulseWatch: 109141 | Liu: 4363


## 3. Gộp, kiểm tra và lưu

In [5]:
df = pd.concat(frames, ignore_index=True)
print(f"{len(df)} cửa sổ | {df.subject_id.nunique()} bệnh nhân | AF {df.label_v7.mean():.1%}")
display(df.groupby(["domain", "class_name"]).size().unstack(fill_value=0))
print("Dòng có đặc trưng NaN (giữ nguyên như bảng gốc):", int(df[FEATURES].isna().any(axis=1).sum()))
assert not df.duplicated(subset=["domain", "subject_id"] + FEATURES).any(), "có cửa sổ trùng"

out = OUT if OVERWRITE else OUT.with_name(OUT.stem + "_rebuilt.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print("Đã lưu:", out.relative_to(ROOT), f"({out.stat().st_size/1e6:.1f} MB)")
if OUT.exists() and out != OUT:
    ref = pd.read_csv(OUT, low_memory=False)
    same = len(ref) == len(df) and np.allclose(ref[FEATURES].to_numpy(), df[FEATURES].to_numpy())
    print("So với bảng chuẩn đang dùng:", "GIỐNG HỆT" if same else "KHÁC, kiểm tra lại đầu vào")

137476 cửa sổ | 248 bệnh nhân | AF 23.3%


class_name,AF,NON_AF,NSR,OTHER,PAC_PVC
domain,,,,,
DEEPBEAT,9317,10525,0,0,0
LIU,2219,0,0,2144,0
MIMIC_PERFORM,2242,1888,0,0,0
PULSEWATCH,18249,0,79544,0,11348


Dòng có đặc trưng NaN (giữ nguyên như bảng gốc): 1


Đã lưu: data\features\v7\healthsense_af_v6c_pacaware_train_rebuilt.csv (35.4 MB)


So với bảng chuẩn đang dùng: KHÁC, kiểm tra lại đầu vào
